# GridSearch & Pipelines

Dado que requerimos probar multitud de parámetros de cara a encontrar de forma empírica aquellos que mejor funcionan, existen medios para automatizar ese proceso y así poder garantizar reproducibilidad en nuestros experimentos.

`GridSearch` es una herramienta de optimización que usamos cuando ajustamos hiperparámetros. Definimos la cuadrícula(grid) de parámetros que queremos buscar y seleccionamos la mejor combinación de parámetros para nuestros datos.


## Método 1
Itera un único algoritmo sobre un conjunto de hiperparámetros, mediante la validación cruzada, iterando con el dataset dividido en train y val para recoger los errores y evaluar la mejor métrica. 

In [1]:
from sklearn import svm, datasets
from sklearn.model_selection import GridSearchCV

iris = datasets.load_iris()

parameters = {
    'kernel': ['linear', 'rbf', 'sigmoid', 'poly'],
    'C': [0.001, 0.1, 0.5, 1, 5, 10, 100],
    'degree': [1,2,3,4,5,6,7],
    'gamma': ['scale', 'auto']
}

svc = svm.SVC()

clf = GridSearchCV(estimator = svc,
                  param_grid = parameters,
                  n_jobs = -1,
                  cv = 10,
                  scoring="accuracy")

clf.fit(iris.data, iris.target)

GridSearchCV(cv=10, estimator=SVC(), n_jobs=-1,
             param_grid={'C': [0.001, 0.1, 0.5, 1, 5, 10, 100],
                         'degree': [1, 2, 3, 4, 5, 6, 7],
                         'gamma': ['scale', 'auto'],
                         'kernel': ['linear', 'rbf', 'sigmoid', 'poly']},
             scoring='accuracy')

Podéis consultar todos los métodos y atributos disponibles en la documentación de [GridSearchCV](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html#gridsearchcv) pero veremos algunos clave para nosotros. Por ejemplo, obtener el modelo ganador:

In [2]:
clf.best_estimator_

SVC(C=0.1, degree=2, gamma='auto', kernel='poly')

Sus parámetros y puntuación obtenida en la métrica objetivo:

In [3]:
print(clf.best_params_)
print(clf.best_score_)

{'C': 0.1, 'degree': 2, 'gamma': 'auto', 'kernel': 'poly'}
0.9866666666666667


In [4]:
clf.score(iris.data, iris.target)

0.9933333333333333

Podemos replicar el modelo empleando los parámetros obtenidos y ver que efectivamente coincide.

In [5]:
from sklearn.model_selection import cross_val_score

clf = svm.SVC(C=0.1, degree=2, gamma='auto', kernel='poly')
scores = cross_val_score(clf, iris.data, iris.target, cv=10)
scores

array([1.        , 0.93333333, 1.        , 1.        , 1.        ,
       1.        , 0.93333333, 1.        , 1.        , 1.        ])

In [6]:
import numpy as np

print(np.mean(scores))

0.9866666666666667


## Método 2

Una forma más eficaz es montar un único gridsearch para iterar con varios modelos ya que no sabemos cual va a resultar con otros hiperparámetros y con la validación cruzada.

In [7]:
# Load data
iris = datasets.load_iris()
X = iris.data
y = iris.target

In [13]:
from sklearn.model_selection import train_test_split 

X_train, X_test, y_train, y_test = train_test_split(X,
                                                    y,
                                                    test_size=0.2,
                                                    random_state=2)

In [26]:
# Load libraries
import numpy as np
from sklearn import datasets
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn import svm

# Set random seed
np.random.seed(0)

pipe = Pipeline(steps=[("scaler", StandardScaler()),
    ('classifier', RandomForestClassifier())
])

logistic_params = {
    'classifier': [LogisticRegression(max_iter=1000, solver='liblinear'), LogisticRegression(max_iter=100, solver='liblinear')],
    'classifier__penalty': ['l1', 'l2']
}

random_forest_params = {
    'scaler': [StandardScaler(), MinMaxScaler()],
    'classifier': [RandomForestClassifier()],
    'classifier__max_depth': [2,3,4]
}

svm_param = {
    'classifier': [svm.SVC(probability=True)],
    'classifier__C': [0.001, 0.1, 0.5, 1, 5, 10, 100],
}

search_space = [
    logistic_params,
    random_forest_params,
    svm_param
]

clf = GridSearchCV(estimator = pipe,
                  param_grid = search_space,
                  cv = 5,
                  n_jobs=-1,
                  verbose=False)

In [27]:
clf

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('scaler', StandardScaler()),
                                       ('classifier',
                                        RandomForestClassifier())]),
             n_jobs=-1,
             param_grid=[{'classifier': [LogisticRegression(max_iter=1000,
                                                            solver='liblinear'),
                                         LogisticRegression(solver='liblinear')],
                          'classifier__penalty': ['l1', 'l2']},
                         {'classifier': [RandomForestClassifier()],
                          'classifier__max_depth': [2, 3, 4],
                          'scaler': [StandardScaler(), MinMaxScaler()]},
                         {'classifier': [SVC(probability=True)],
                          'classifier__C': [0.001, 0.1, 0.5, 1, 5, 10, 100]}],
             verbose=False)

In [28]:
clf.fit(X_train, y_train)

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('scaler', StandardScaler()),
                                       ('classifier',
                                        RandomForestClassifier())]),
             n_jobs=-1,
             param_grid=[{'classifier': [LogisticRegression(max_iter=1000,
                                                            solver='liblinear'),
                                         LogisticRegression(solver='liblinear')],
                          'classifier__penalty': ['l1', 'l2']},
                         {'classifier': [RandomForestClassifier()],
                          'classifier__max_depth': [2, 3, 4],
                          'scaler': [StandardScaler(), MinMaxScaler()]},
                         {'classifier': [SVC(probability=True)],
                          'classifier__C': [0.001, 0.1, 0.5, 1, 5, 10, 100]}],
             verbose=False)

Vemos que aunque nuestro pipeline tenga ciertas opciones definidas, explora las parametrizaciones ofrecidas donde hay otros modelos. Así, obtendremos la mejor combinación de preprocesado, modelo y parámteros a utilizar.

In [29]:
print(clf.best_estimator_)
print(clf.best_score_)
print(clf.best_params_)

Pipeline(steps=[('scaler', StandardScaler()),
                ('classifier', RandomForestClassifier(max_depth=4))])
0.9666666666666668
{'classifier': RandomForestClassifier(), 'classifier__max_depth': 4, 'scaler': StandardScaler()}


In [30]:
clf.best_estimator_.predict(X_test)

array([0, 0, 2, 0, 0, 2, 0, 2, 2, 0, 0, 0, 0, 0, 1, 1, 0, 1, 2, 1, 2, 1,
       2, 1, 1, 0, 0, 2, 0, 2])

In [31]:
clf.best_estimator_.score(X_test,y_test)

0.9666666666666667

In [32]:
clf.best_estimator_.predict_proba(X_test)

array([[1.00000000e+00, 0.00000000e+00, 0.00000000e+00],
       [1.00000000e+00, 0.00000000e+00, 0.00000000e+00],
       [0.00000000e+00, 7.70614217e-02, 9.22938578e-01],
       [9.95000000e-01, 5.00000000e-03, 0.00000000e+00],
       [1.00000000e+00, 0.00000000e+00, 0.00000000e+00],
       [0.00000000e+00, 3.72366618e-01, 6.27633382e-01],
       [9.80000000e-01, 2.00000000e-02, 0.00000000e+00],
       [0.00000000e+00, 1.93693694e-03, 9.98063063e-01],
       [0.00000000e+00, 2.70270270e-04, 9.99729730e-01],
       [9.95000000e-01, 5.00000000e-03, 0.00000000e+00],
       [1.00000000e+00, 0.00000000e+00, 0.00000000e+00],
       [1.00000000e+00, 0.00000000e+00, 0.00000000e+00],
       [1.00000000e+00, 0.00000000e+00, 0.00000000e+00],
       [1.00000000e+00, 0.00000000e+00, 0.00000000e+00],
       [5.02941176e-02, 9.20236907e-01, 2.94689755e-02],
       [0.00000000e+00, 9.62030692e-01, 3.79693085e-02],
       [9.55000000e-01, 4.16666667e-02, 3.33333333e-03],
       [2.94117647e-04, 9.96297

## Método 3

Otro uso puede ser la construcción de pipelines (tuberías) específicos para cada tipo de modelo.

In [34]:
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.feature_selection import SelectKBest
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score

import pandas as pd
import numpy as np

from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

In [35]:
reg_log = Pipeline(steps = [
    ("imputer", SimpleImputer()),
    ("scaler", StandardScaler()),
    ("reglog", LogisticRegression())
])
reg_log_param = {
    "imputer__strategy": ['mean', 'median'],
    "reglog__penalty": ['l1', 'l2'],
    "reglog__C": np.logspace(0, 4, 10)
}

In [36]:

rand_forest = RandomForestClassifier()
rand_forest_param = {
    "n_estimators": [10, 100, 1000],
    "max_features": [1,2,3]
}


svm = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("selectkbest", SelectKBest()),
    ("svm", SVC())
])


svm_param = {
    'selectkbest__k': [2, 3, 4],
    'svm__kernel': ['linear', 'rbf', 'sigmoid', 'poly'],
    'svm__C': [0.001, 0.1, 0.5, 1, 5, 10, 100],
    'svm__degree': [1,2,3,4],
    'svm__gamma': ['scale', 'auto']
}


gs_reg_log = GridSearchCV(reg_log,
                         reg_log_param,
                         cv = 10,
                         scoring = 'accuracy',
                         verbose = 1,
                         n_jobs = -1,
                         error_score=0)

gs_rand_forest = GridSearchCV(rand_forest,
                         rand_forest_param,
                         cv = 10,
                         scoring = 'accuracy',
                         verbose = 1,
                         n_jobs = -1,
                         error_score=0)

gs_svm = GridSearchCV(svm,
                         svm_param,
                         cv = 10,
                         scoring = 'accuracy',
                         verbose = 1,
                         n_jobs = -1,
                         error_score=0)

grids = {"gs_reg_log": gs_reg_log,
        "gs_rand_forest": gs_rand_forest,
        "gs_svm": gs_svm}

In [37]:
from sklearn.model_selection import train_test_split

X = iris.data
y = iris.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [38]:
import warnings
warnings.filterwarnings(action="ignore")

for nombre, grid_search in grids.items():
    grid_search.fit(X_train, y_train)

Fitting 10 folds for each of 40 candidates, totalling 400 fits
Fitting 10 folds for each of 9 candidates, totalling 90 fits
Fitting 10 folds for each of 672 candidates, totalling 6720 fits


In [39]:
print("Mejor puntuación = ", gs_reg_log.best_score_)
print("Mejores parámetros = ", gs_reg_log.best_params_)
print("Mejor modelo = ", gs_reg_log.best_estimator_)

Mejor puntuación =  0.9583333333333334
Mejores parámetros =  {'imputer__strategy': 'mean', 'reglog__C': np.float64(7.742636826811269), 'reglog__penalty': 'l2'}
Mejor modelo =  Pipeline(steps=[('imputer', SimpleImputer()), ('scaler', StandardScaler()),
                ('reglog',
                 LogisticRegression(C=np.float64(7.742636826811269)))])


In [40]:
print("Mejor puntuación = ", gs_rand_forest.best_score_)
print("Mejores parámetros = ", gs_rand_forest.best_params_)
print("Mejor modelo = ", gs_rand_forest.best_estimator_)

Mejor puntuación =  0.9416666666666667
Mejores parámetros =  {'max_features': 1, 'n_estimators': 10}
Mejor modelo =  RandomForestClassifier(max_features=1, n_estimators=10)


In [41]:
print("Mejor puntuación = ", gs_svm.best_score_)
print("Mejores parámetros = ", gs_svm.best_params_)
print("Mejor modelo = ", gs_svm.best_estimator_)

Mejor puntuación =  0.9666666666666668
Mejores parámetros =  {'selectkbest__k': 4, 'svm__C': 5, 'svm__degree': 1, 'svm__gamma': 'scale', 'svm__kernel': 'linear'}
Mejor modelo =  Pipeline(steps=[('scaler', StandardScaler()), ('selectkbest', SelectKBest(k=4)),
                ('svm', SVC(C=5, degree=1, kernel='linear'))])


In [42]:
best_grids = [(i, j.best_score_) for i, j in grids.items()]

best_grids = pd.DataFrame(best_grids, columns=["Grid", "Best score"]).sort_values(by="Best score", ascending=False)
best_grids

,Grid,Best score
2,gs_svm,0.966667
0,gs_reg_log,0.958333
1,gs_rand_forest,0.941667


Nos quedamos con el SVM entonces. ¿Cierto?

In [43]:
preds = gs_svm.best_estimator_.predict(X_test)
accuracy_score(y_test, preds)

0.9666666666666667

In [44]:
preds = gs_reg_log.best_estimator_.predict(X_test)
accuracy_score(y_test, preds)

1.0

In [45]:
preds = gs_rand_forest.best_estimator_.predict(X_test)
accuracy_score(y_test, preds)

1.0

 Tanto la regresión logísitca(pipeline) como el random forest son los modelos que mejor generalizan. Después de tanto entrenar, será mejor que guardemos el modelo para poder emplearlo más adelante.

In [48]:
best_model = gs_svm.best_estimator_
best_model

Pipeline(steps=[('scaler', StandardScaler()), ('selectkbest', SelectKBest(k=4)),
                ('svm', SVC(C=5, degree=1, kernel='linear'))])

In [49]:
import pickle

filename = 'mi_modelo'

with open(filename, 'wb') as archivo_salida:
    pickle.dump(best_model, archivo_salida)

Esto nos permite volver a cargarlo en cualquier otro momento para ser utilizado.

In [50]:
with open(filename, 'rb') as archivo_entrada:
    modelo_importado = pickle.load(archivo_entrada)

In [51]:
modelo_importado

Pipeline(steps=[('scaler', StandardScaler()), ('selectkbest', SelectKBest(k=4)),
                ('svm', SVC(C=5, degree=1, kernel='linear'))])

In [53]:
print("Exactitud del modelo importado = ", np.round(modelo_importado.score(X_test, y_test)*100, 2), "%")

Exactitud del modelo importado =  96.67 %


In [54]:
modelo_importado.predict(X_test)

array([1, 0, 2, 1, 1, 0, 1, 2, 2, 1, 2, 0, 0, 0, 0, 1, 2, 1, 1, 2, 0, 2,
       0, 2, 2, 2, 2, 2, 0, 0])

Ya hemos escogido modelo gracias a los datos de validación. Ahora habría que entrenar el modelo con TODOS los datos de train.

## RandomSearch
El problema que tiene el GridSearchCV es que computacionalmente es muy costoso cuando el espacio dimensional de los hiperparámetros es grande.

Mediante el RandomSearch no se prueban todas las combinaciones, sino unas cuantas de manera aleatoria. Funciona bien con datasets con pocas features. Incluso [hay papers](https://www.jmlr.org/papers/v13/bergstra12a.html) que aseguran que es más eficiente RandomSearch frente a GridSearch

![imagen](https://miro.medium.com/proxy/1*ZTlQm_WRcrNqL-nLnx6GJA.png)

In [55]:
from sklearn.model_selection import RandomizedSearchCV

reg_log = Pipeline(steps=[
                          ("imputer",SimpleImputer()),
                          ("scaler",StandardScaler()),
                          ("reglog",LogisticRegression())
                         ])

reg_log_param = {    
                 "imputer__strategy": ['mean', 'median', 'most_frequent'],
                 "reglog__penalty": ["l1","l2"], 
                 "reglog__C": np.logspace(0, 4, 10)
                }


search = RandomizedSearchCV(reg_log,
                           reg_log_param,
                           n_iter = 50,
                           scoring='accuracy',
                           n_jobs=-1,
                           cv=10)

# execute search
result = search.fit(X_train, y_train)

# summarize result
print('Best Score: %s' % result.best_score_)
print('Best Hyperparameters: %s' % result.best_params_)
print('Best Estimator: %s' % result.best_estimator_)

Best Score: 0.9583333333333334
Best Hyperparameters: {'reglog__penalty': 'l2', 'reglog__C': np.float64(7.742636826811269), 'imputer__strategy': 'most_frequent'}
Best Estimator: Pipeline(steps=[('imputer', SimpleImputer(strategy='most_frequent')),
                ('scaler', StandardScaler()),
                ('reglog',
                 LogisticRegression(C=np.float64(7.742636826811269)))])


In [57]:
search

RandomizedSearchCV(cv=10,
                   estimator=Pipeline(steps=[('imputer', SimpleImputer()),
                                             ('scaler', StandardScaler()),
                                             ('reglog', LogisticRegression())]),
                   n_iter=50, n_jobs=-1,
                   param_distributions={'imputer__strategy': ['mean', 'median',
                                                              'most_frequent'],
                                        'reglog__C': array([1.00000000e+00, 2.78255940e+00, 7.74263683e+00, 2.15443469e+01,
       5.99484250e+01, 1.66810054e+02, 4.64158883e+02, 1.29154967e+03,
       3.59381366e+03, 1.00000000e+04]),
                                        'reglog__penalty': ['l1', 'l2']},
                   scoring='accuracy')

In [56]:
# summarize result
print('Best Score: %s' % result.best_score_)
print('Best Hyperparameters: %s' % result.best_params_)
print('Best Estimator: %s' % result.best_estimator_)

Best Score: 0.9583333333333334
Best Hyperparameters: {'reglog__penalty': 'l2', 'reglog__C': np.float64(7.742636826811269), 'imputer__strategy': 'most_frequent'}
Best Estimator: Pipeline(steps=[('imputer', SimpleImputer(strategy='most_frequent')),
                ('scaler', StandardScaler()),
                ('reglog',
                 LogisticRegression(C=np.float64(7.742636826811269)))])


## Extra: Optuna

En los últimos años ha proliferado mucho el uso de [Optuna](https://optuna.org/) una librería específica para la búsqueda de hiperparámetros. Más allá del GridSearch y RandomSearch implementa búsquedas algo más eficientes en tiempo y con más opciones de ajuste.

In [1]:
# !pip install optuna optuna-dashboard

Debemos de crear un estudio, una sesión en la que registraremos todos nuestros intentos.

In [21]:
import optuna

study = optuna.create_study(
    storage="sqlite:///db.sqlite3",  # Almacenamiento
    study_name="quadratic-simple",
    load_if_exists=True
)

[I 2025-06-24 10:22:44,658] A new study created in RDB with name: quadratic-simple


In [22]:
def objective(trial):
    x = trial.suggest_float('x', -10, 10)
    return (x - 2) ** 2

study.optimize(objective, n_trials=100)

[I 2025-06-24 10:22:45,923] Trial 0 finished with value: 68.2494733279301 and parameters: {'x': -6.2613239452239196}. Best is trial 0 with value: 68.2494733279301.
[I 2025-06-24 10:22:45,944] Trial 1 finished with value: 13.308060248804798 and parameters: {'x': -1.648021415617622}. Best is trial 1 with value: 13.308060248804798.
[I 2025-06-24 10:22:45,969] Trial 2 finished with value: 137.91422345176318 and parameters: {'x': -9.743688664630172}. Best is trial 1 with value: 13.308060248804798.
[I 2025-06-24 10:22:45,990] Trial 3 finished with value: 57.935005678581376 and parameters: {'x': 9.611504823527433}. Best is trial 1 with value: 13.308060248804798.
[I 2025-06-24 10:22:46,012] Trial 4 finished with value: 0.5956510438471903 and parameters: {'x': 2.7717843246964726}. Best is trial 4 with value: 0.5956510438471903.
[I 2025-06-24 10:22:46,032] Trial 5 finished with value: 31.30004522280443 and parameters: {'x': 7.594644333896877}. Best is trial 4 with value: 0.5956510438471903.
[I 2

In [23]:
study.best_params

{'x': 2.0008866783291475}

Ahora probad a ejecutar la siguiente celda o ejecutarlo en un terminal con el entorno donde tengáis optuna instalado. El dashboard os da un cuadro de mando de los experimentos realizados. 

In [24]:
# !optuna-dashboard sqlite:///db.sqlite3

También existe una [extensión de VSCode](https://marketplace.visualstudio.com/items?itemName=Optuna.optuna-dashboard#overview) para que no tengáis que lanzar el comando anterior y os sea más cómodo.

Una de las bondades de Optuna es que nos permite explorar de forma eficiente el espacio de solución, con opciones más allá de Grid Y Random:

* Grid Search
* Random Search
* Tree-structured Parzen Estimator algorithm
* CMA-ES based algorithm
* Gaussian process-based algorithm

https://optuna.readthedocs.io/en/stable/tutorial/10_key_features/003_efficient_optimization_algorithms.html

In [1]:
from sklearn import datasets

# Dataset
iris = datasets.load_breast_cancer()

In [ ]:
import optuna
from sklearn.model_selection import train_test_split

# Train/test
X = iris.data
y = iris.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

study = optuna.create_study(
    storage="sqlite:///db.sqlite3",  # Almacenamiento
    study_name="modelo-cancer",
    direction='maximize',
    sampler=optuna.samplers.TPESampler(), #CmaEsSampler(), # Probad con otros
    load_if_exists=True
)

Otra de las ventajas es que podemos pedir a optuna que _sugiera_ los parámetros: https://optuna.readthedocs.io/en/stable/reference/samplers/generated/optuna.samplers.BaseSampler.html#optuna-samplers-basesampler

In [10]:
import xgboost as xgb
from sklearn.metrics import roc_auc_score

# Función objetivo
def objective(trial):
    # 2. Suggest values of the hyperparameters using a trial object.
    param = {
        "objective": "binary:logistic",
        "booster": trial.suggest_categorical("booster", ["gbtree", "gblinear", "dart"]),
        "lambda": trial.suggest_float("lambda", 1e-8, 1.0, log=True),
        "alpha": trial.suggest_float("alpha", 1e-8, 1.0, log=True),
        "subsample": trial.suggest_float("subsample", 0.2, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.2, 1.0),
    }

    clf = xgb.XGBClassifier(
        **param,
        eval_metric="auc"
    )
    clf.fit(X_train, y_train)
    
    pred_proba = clf.predict_proba(X_test)[:,1]

    return roc_auc_score(y_true=y_test, y_score=pred_proba) 

In [11]:
# Lanzamos el estudio
study.optimize(objective, n_trials=100)

[I 2025-06-24 10:39:01,934] Trial 100 finished with value: 0.9967245332459875 and parameters: {'booster': 'dart', 'lambda': 0.0010643312477079025, 'alpha': 4.117617943680468e-07, 'subsample': 0.5132148618356577, 'colsample_bytree': 0.6175032683098713}. Best is trial 45 with value: 0.9983622666229939.
[I 2025-06-24 10:39:02,376] Trial 101 finished with value: 0.9937766131673763 and parameters: {'booster': 'dart', 'lambda': 0.0014044145920427658, 'alpha': 1.6943033569447156e-05, 'subsample': 0.5428199498371983, 'colsample_bytree': 0.6147129825301785}. Best is trial 45 with value: 0.9983622666229939.
[I 2025-06-24 10:39:02,914] Trial 102 finished with value: 0.996069439895185 and parameters: {'booster': 'dart', 'lambda': 0.0005687058097616536, 'alpha': 3.4488740069916618e-06, 'subsample': 0.4619008496963548, 'colsample_bytree': 0.5688584275391213}. Best is trial 45 with value: 0.9983622666229939.
[I 2025-06-24 10:39:03,309] Trial 103 finished with value: 0.9895185063871602 and parameters:

Y permite cortar el proceso si vemos que no va por buen camino... mediante los pruners: https://optuna.readthedocs.io/en/stable/tutorial/10_key_features/003_efficient_optimization_algorithms.html#pruning-algorithms

In [12]:
study.best_params

{'booster': 'dart',
 'lambda': 1.466988045387824e-07,
 'alpha': 0.012094444223988542,
 'subsample': 0.6978903505246865,
 'colsample_bytree': 0.8852436642003466}

In [8]:
study.best_trial

FrozenTrial(number=45, state=1, values=[0.9983622666229939], datetime_start=datetime.datetime(2025, 6, 24, 10, 37, 18, 257652), datetime_complete=datetime.datetime(2025, 6, 24, 10, 37, 18, 308346), params={'booster': 'gblinear', 'lambda': 2.5998835120717873e-05, 'alpha': 0.003052613738897168, 'subsample': 0.7835900586350717, 'colsample_bytree': 0.4826972282871704}, user_attrs={}, system_attrs={'cma:generation': 5}, intermediate_values={}, distributions={'booster': CategoricalDistribution(choices=('gbtree', 'gblinear', 'dart')), 'lambda': FloatDistribution(high=1.0, log=True, low=1e-08, step=None), 'alpha': FloatDistribution(high=1.0, log=True, low=1e-08, step=None), 'subsample': FloatDistribution(high=1.0, log=False, low=0.2, step=None), 'colsample_bytree': FloatDistribution(high=1.0, log=False, low=0.2, step=None)}, trial_id=46, value=None)